In [4]:
import polars as pl
from data_engineering.flights.flight_type import get_flights
from datetime import date
new_file = '/Volumes/T2-SSD/planequery/data/tests/test_pia/test_real_flights_4/test_pia_test_real_flights_4_20260826_123248/pia_mapping_audit.parquet'
old_file = '/Volumes/T2-SSD/planequery/data/tests/test_pia/pia_tester/test_pia_pia_tester_20260821_004941/pia_mapping_audit.parquet'
df = pl.read_parquet(new_file, glob=False)

In [5]:
df = df.unique(subset="observed_icao").filter(pl.col("predicted_icao").is_not_null())
df = df.select("observed_icao", "predicted_icao", "real_icao", "correct")
df

observed_icao,predicted_icao,real_icao,correct
str,str,str,bool
"""a03d98""","""a00529""",null,null
"""a0950d""","""a58191""",null,null
"""a09227""","""a31b62""",null,null
"""a03d00""","""a5b3dc""",null,null
"""a03cb2""","""a1f3e1""",null,null
…,…,…,…
"""a09263""","""a70b8c""",null,null
"""a03c7b""","""adf396""",null,null
"""a0feaf""","""a500fe""","""a901cd""",false


In [6]:
df_flights = get_flights(date(2026,1,1), date(2026,7,1))

In [7]:
observed_icaos = df["observed_icao"].to_list()
predicted_real_icaos = df["predicted_icao"].to_list()
real_icaos = df["real_icao"].to_list()
correct = df["correct"].to_list()
n_switches_list = []
for observed_icao, predicted_real_icao, real_icao, is_correct in zip(observed_icaos, predicted_real_icaos, real_icaos, correct):
    df_flights_filtered = df_flights.filter(pl.col("icao").is_in([predicted_real_icao, observed_icao])).sort("takeoff_time")
    icao_sequence = df_flights_filtered["icao"].to_list()
    n_switches = sum(1 for i in range(1, len(icao_sequence)) if icao_sequence[i] != icao_sequence[i - 1])
    n_switches_list.append(n_switches)

df_mapping = df.with_columns(pl.Series(n_switches_list).alias("n_switches"))
df_mapping

observed_icao,predicted_icao,real_icao,correct,n_switches
str,str,str,bool,i64
"""a03d98""","""a00529""",null,null,1
"""a0950d""","""a58191""",null,null,1
"""a09227""","""a31b62""",null,null,1
"""a03d00""","""a5b3dc""",null,null,1
"""a03cb2""","""a1f3e1""",null,null,1
…,…,…,…,…
"""a09263""","""a70b8c""",null,null,1
"""a03c7b""","""adf396""",null,null,1
"""a0feaf""","""a500fe""","""a901cd""",false,1


In [8]:
df_mapping.sort("n_switches", descending=True)

observed_icao,predicted_icao,real_icao,correct,n_switches
str,str,str,bool,i64
"""a03c06""","""a97b71""","""a16acc""",false,6
"""a1010d""","""a2cc0f""",null,null,2
"""a0ff10""","""a15fe4""",null,null,2
"""a03d98""","""a00529""",null,null,1
"""a0950d""","""a58191""",null,null,1
…,…,…,…,…
"""a09263""","""a70b8c""",null,null,1
"""a03c7b""","""adf396""",null,null,1
"""a0feaf""","""a500fe""","""a901cd""",false,1


In [9]:
COLUMNS = ["icao", "pia", "takeoff_time", "takeoff_airport_ident", "landing_time", "landing_airport_ident", "callsign"]
with pl.Config(tbl_rows=-1):
    display(df_flights.filter(pl.col("icao").is_in(["a03d26", "a4a3e8"])).select(COLUMNS).sort("takeoff_time"))

icao,pia,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,callsign
str,bool,datetime[ms],str,datetime[ms],str,str


In [10]:
COLUMNS = [
    "icao",
    "pia",
    "takeoff_time",
    "takeoff_airport_ident",
    "landing_time",
    "landing_airport_ident",
    "callsign",
]

flights = (
    df_flights
    .filter(pl.col("icao").is_in(["a03d26", "a4a3e8"]))
    .sort("takeoff_time")
    .with_columns(
        pl.col("icao")
        .ne(pl.col("icao").shift())
        .fill_null(False)
        .alias("_icao_changed")
    )
)

# Two rows before and two rows after each ICAO boundary.
around_change = flights.filter(
    pl.any_horizontal(
        pl.col("_icao_changed").shift(-2, fill_value=False),
        pl.col("_icao_changed").shift(-1, fill_value=False),
        pl.col("_icao_changed"),
        pl.col("_icao_changed").shift(1, fill_value=False),
    )
).select(COLUMNS)

with pl.Config(tbl_rows=-1):
    display(around_change)

icao,pia,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,callsign
str,bool,datetime[ms],str,datetime[ms],str,str
